In [ ]:
%load_ext autoreload
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from imblearn.under_sampling import RandomUnderSampler
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.metrics import classification_report
from torch.autograd import detect_anomaly
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, \
    GenerationConfig
import google.generativeai as genai
from google.generativeai import types
import os
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import google.api_core.exceptions
from dotenv import load_dotenv
from openai import OpenAI
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from accelerate import Accelerator
import TextMiningBasedSatdDetectorModel


In [ ]:
load_dotenv()
DEFAULT_DETECTION_CLASS = 'no'

# Detection Dataset

In [ ]:
detect_train_df = pd.read_csv('../data/detect_train.csv')
detect_train_dataset = Dataset.from_pandas(detect_train_df)

detect_test_df = pd.read_csv('../data/detect_test.csv')
detect_test_dataset = Dataset.from_pandas(detect_test_df)


# SATD detector training dataset preparation

In [ ]:
BASE_SATD_DETECTOR_DIRECTORY = os.getenv('BASE_SATD_DETECTOR_DIRECTORY')
os.makedirs(BASE_SATD_DETECTOR_DIRECTORY, exist_ok=True)
os.makedirs(os.path.join(BASE_SATD_DETECTOR_DIRECTORY, 'models'), exist_ok=True)
detect_train_df['text'].str.replace('\n', '\t').to_csv(f'{BASE_SATD_DETECTOR_DIRECTORY}/comments.txt', index=False, header=False)
detect_train_df['label'].str.lower().map({'yes': 'Yes', 'no': 'No'}).to_csv(f'{BASE_SATD_DETECTOR_DIRECTORY}/labels.txt', index=False, header=False)
detect_train_df.assign(project='train')['project'].to_csv(f'{BASE_SATD_DETECTOR_DIRECTORY}/projects.txt', index=False, header=False)

In [ ]:
pretrained_satd_detector = TextMiningBasedSatdDetectorModel('detect', 'pretrained-liu-detector', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
pretrained_satd_detector.fit(detect_train_dataset)
pretrained_satd_detector.predict(detect_test_dataset)

In [ ]:
trained_detector = TextMiningBasedSatdDetectorModel('detect', 'trained-liu-detector', {'yes', 'no'}, DEFAULT_DETECTION_CLASS, retrain=True)
trained_detector.fit(detect_train_dataset)
trained_detector.predict(detect_test_dataset)